# Notebook 07 — ODE State Layout and Simulation

This notebook explains the full ODE state vector, the equations that govern each
compartment, and how to run a forward simulation with `simulate()` from
`phoscrosstalk.simulation`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = PROJECT_ROOT / "notebooks" / "sample_data"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1  ODE state vector

The model state at time *t* is a single flat vector:

```
y(t) = [ R_rna(K) | S(K) | A(K) | Kdyn(M) | p(N) ]
         ─────────  ──────  ──────  ─────────  ─────
         mRNA       sig.    abund.  kinase     phospho
```

| Slice | Symbol | Shape | Meaning |
|---|---|---|---|
| `y[0:K]` | `R_rna` | `(K,)` | mRNA level per protein |
| `y[K:2K]` | `S` | `(K,)` | Protein signalling state |
| `y[2K:3K]` | `A` | `(K,)` | Protein abundance |
| `y[3K:3K+M]` | `Kdyn` | `(M,)` | Dynamic kinase activity |
| `y[3K+M:]` | `p` | `(N,)` | Phosphosite occupancy |

For sample data (K=3, M=2, N=9): total dim = 9 + 2 + 9 = **20**.

In [ ]:
from phoscrosstalk.config import ModelDims
from phoscrosstalk.data_loader import (
    load_site_data, load_rna_data, load_kinase_site_matrix,
    load_tf_network, build_tf_prot_weights, apply_scaling, row_normalize,
)

TIMEPOINTS = list(range(1, 15))   # 14 time points x1..x14

# ── phosphosite + protein abundance ─────────────────────────────────────────
sites, proteins, site_prot_idx, positions, t_phos, Y, A_data, A_proteins = \
    load_site_data(str(SAMPLE_DIR / "protephospho.csv"), TIMEPOINTS)

K = len(proteins)
N = len(sites)
T = len(t_phos)
print(f"proteins : {proteins}  (K={K})")
print(f"sites    : {sites}  (N={N})")
print(f"t_phos   : {t_phos}  (T={T})")
print(f"Y        : {Y.shape}   (N × T  phosphosite data)")
print(f"A_data   : {A_data.shape}  (K × T  protein abundance)")

# ── mRNA ─────────────────────────────────────────────────────────────────────
gene_ids, t_rna, rna_matrix = load_rna_data(
    str(SAMPLE_DIR / "mrna.csv"), timepoints=TIMEPOINTS
)
print(f"gene_ids : {gene_ids}  (n_genes={len(gene_ids)})")
print(f"rna_matrix: {rna_matrix.shape}  (n_genes × T)")

# ── kinase-site matrix ───────────────────────────────────────────────────────
K_site_kin, kinases = load_kinase_site_matrix(
    str(SAMPLE_DIR / "kinase_sites.tsv"), sites
)
M = len(kinases)
print(f"kinases  : {kinases}  (M={M})")
print(f"K_site_kin: {K_site_kin.shape}  (N × M)")

# ── kinase → protein index ───────────────────────────────────────────────────
kin_to_prot_idx = np.array([proteins.index(k) for k in kinases], dtype=int)
print(f"kin_to_prot_idx: {kin_to_prot_idx}")

# ── TF network ───────────────────────────────────────────────────────────────
tf_net = load_tf_network(str(SAMPLE_DIR / "tf_mrna.csv"), gene_ids=gene_ids)
tf_prot_weights = build_tf_prot_weights(tf_net, gene_ids, proteins)
print(f"tf_prot_weights: {tf_prot_weights.shape}  (K × n_genes)")

# ── scaled data ──────────────────────────────────────────────────────────────
P_scaled, _, _  = apply_scaling(Y)
A_scaled, _, _  = apply_scaling(A_data)
dims = ModelDims(K=K, M=M, N=N)

In [ ]:
state_dim = 3*K + M + N
print(f"State dim = 3×{K} + {M} + {N} = {state_dim}")

# Annotate each block
slices = [
    ("R_rna[0:K]",         0,      K),
    ("S[K:2K]",            K,      2*K),
    ("A[2K:3K]",           2*K,    3*K),
    ("Kdyn[3K:3K+M]",      3*K,    3*K+M),
    ("p[3K+M:3K+M+N]",     3*K+M,  state_dim),
]
for name, lo, hi in slices:
    print(f"  y[{lo}:{hi}]  ({hi-lo:2d} dims)  ← {name}")


## 2  ODE equations (condensed)

Each compartment obeys a first-order ODE:

```
dR_rna / dt  =  k_act(t)  −  k_deact · R_rna
dS     / dt  =  k_act(t)  −  k_deact · S
dA     / dt  =  s_prod(t) −  d_deg   · A
dKdyn  / dt  =  α · (K_site_kin^T @ p) · kK_act  −  kK_deact · Kdyn
dp_i   / dt  =  ( Σ_m Kdyn_m · K_site_kin[i,m]
                  + β_g · (Cg @ p)[i]
                  + β_l · (Cl @ p)[i] ) · gate_i  −  k_off_i · p_i
```

`gate_i` encodes the sequential/distributive/random mechanism.
`Cg`, `Cl` are global/local crosstalk matrices.

## 3  Build all simulation matrices

In [ ]:
from phoscrosstalk.optimization import create_bounds, build_parameter_labels

# Placeholder crosstalk (no prior network)
Cg = np.zeros((N, N))
Cl = np.zeros((N, N))

# Kinase feedback matrix: (M, N) row-normalised
R_kin = row_normalize(K_site_kin.T)

# Kinase-network Laplacian (no prior network structure)
L_alpha = np.zeros((M, M))

# Receptor masks (no receptor-specific forcing)
receptor_mask_prot = np.zeros(K)
receptor_mask_kin  = np.zeros(M)

print("Cg:", Cg.shape, "  Cl:", Cl.shape)
print("R_kin:", R_kin.shape, "  L_alpha:", L_alpha.shape)
print("K_site_kin:", K_site_kin.shape)


## 4  Build theta at midpoint of log-space bounds

In [ ]:
xl, xu, dim = create_bounds(K, M, N)
theta_mid = 0.5 * (xl + xu)
print(f"theta dim: {dim}")
labels = build_parameter_labels(K, M, N)
print("First 10 labels:", labels[:10])
print("theta_mid (first 10):", theta_mid[:10].round(3))


## 5  Build derived-rate closures

In [ ]:
from phoscrosstalk.derived_rates import make_k_act_fn, make_s_prod_fn

k_act_fn = make_k_act_fn(
    t_rna=t_rna, rna_data=rna_matrix,
    tf_prot_weights=tf_prot_weights, K=K,
)

s_prod_fn = make_s_prod_fn(
    t_protein=t_phos,
    Y_data=P_scaled,
    R_kin_site=row_normalize(K_site_kin.T),
    kin_to_prot_idx=kin_to_prot_idx,
    K=K, M=M,
)
print("k_act_fn and s_prod_fn built successfully")


## 6  Run the forward simulation

In [ ]:
from phoscrosstalk.simulation import simulate

P_sim, A_sim = simulate(
    t_arr=t_phos,
    P_data0=P_scaled,
    A_data0=A_scaled,
    theta=theta_mid,
    Cg=Cg, Cl=Cl,
    site_prot_idx=site_prot_idx,
    K_site_kin=K_site_kin,
    R=R_kin,
    L_alpha=L_alpha,
    kin_to_prot_idx=kin_to_prot_idx,
    receptor_mask_prot=receptor_mask_prot,
    receptor_mask_kin=receptor_mask_kin,
    mechanism="dist",
    full_output=False,
    k_act_fn=k_act_fn,
    s_prod_fn=s_prod_fn,
)
print("P_sim:", P_sim.shape, "  (N sites × T)")
print("A_sim:", A_sim.shape, "  (K proteins × T)")
print("Any NaN in P_sim?", np.any(np.isnan(P_sim)))
print("Any NaN in A_sim?", np.any(np.isnan(A_sim)))


## 7  Full output: S_sim and Kdyn_sim

Set `full_output=True` to also return the protein signalling state `S_sim` (K×T)
and dynamic kinase activity `Kdyn_sim` (M×T).

In [ ]:
P_sim2, A_sim2, S_sim, Kdyn_sim = simulate(
    t_arr=t_phos, P_data0=P_scaled, A_data0=A_scaled,
    theta=theta_mid, Cg=Cg, Cl=Cl,
    site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
    L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
    receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
    mechanism="dist", full_output=True,
    k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
)
print("S_sim:    ", S_sim.shape,    "  protein signalling state")
print("Kdyn_sim: ", Kdyn_sim.shape, "  kinase dynamic activity")


## 8  Plot P_sim vs P_data for all phosphosites

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
axes = axes.ravel()
for i, site in enumerate(sites):
    ax = axes[i]
    ax.plot(t_phos, P_scaled[i], "o--", ms=4, label="data",  color="tab:blue")
    ax.plot(t_phos, P_sim[i],    "-",   lw=2, label="sim",   color="tab:orange")
    ax.set_title(site, fontsize=9)
    if i == 0:
        ax.legend(fontsize=7)
fig.suptitle("Phosphosite simulation vs data  (theta_mid)", y=1.01)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_phospho_sim_vs_data.png", dpi=100)
plt.show()
print("Saved 07_phospho_sim_vs_data.png")


## 9  Plot A_sim vs A_data for all proteins

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for p_idx, pname in enumerate(proteins):
    ax = axes[p_idx]
    ax.plot(t_phos, A_scaled[p_idx], "o--", ms=4, label="data",  color="tab:blue")
    ax.plot(t_phos, A_sim[p_idx],    "-",   lw=2, label="sim",   color="tab:orange")
    ax.set_title(pname); ax.set_xlabel("Time index")
    if p_idx == 0:
        ax.set_ylabel("Abundance (scaled)"); ax.legend()
fig.suptitle("Protein abundance simulation vs data  (theta_mid)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_protein_sim_vs_data.png", dpi=100)
plt.show()
print("Saved 07_protein_sim_vs_data.png")


## 10  Mechanism variants: `dist`, `seq`, `rand`

| Mechanism | Description |
|---|---|
| `dist` | Distributive: each site phosphorylated independently |
| `seq` | Sequential: requires previous site to be occupied first |
| `rand` | Random: no ordering constraint |

`compute_prev_site_idx()` determines the predecessor site for sequential gating.

In [ ]:
from phoscrosstalk.mechanisms import compute_prev_site_idx

prev_idx = compute_prev_site_idx(np.array(site_prot_idx), N)
print("site_prot_idx:", np.array(site_prot_idx))
print("prev_site_idx:", prev_idx)
print("(−1 = first site for that protein — no predecessor)")


In [ ]:
# Compare mechanisms
for mech in ["dist", "seq", "rand"]:
    P_m, _ = simulate(
        t_arr=t_phos, P_data0=P_scaled, A_data0=A_scaled,
        theta=theta_mid, Cg=Cg, Cl=Cl,
        site_prot_idx=site_prot_idx, K_site_kin=K_site_kin, R=R_kin,
        L_alpha=L_alpha, kin_to_prot_idx=kin_to_prot_idx,
        receptor_mask_prot=receptor_mask_prot, receptor_mask_kin=receptor_mask_kin,
        mechanism=mech, k_act_fn=k_act_fn, s_prod_fn=s_prod_fn,
    )
    diff = float(np.nanmean(np.abs(P_m - P_sim)))
    print(f"mechanism={mech!r}   mean |ΔP_sim vs dist|: {diff:.6f}")


## 11  ODE solver: Diffrax / Tsit5

The ODE is integrated with **Diffrax** (JAX-native ODE library):

- **Solver**: `Tsit5` — 4th/5th-order Runge-Kutta with adaptive stepping
- **Tolerance**: `rtol=1e-6`, `atol=1e-9`
- **Adjoint**: `RecursiveCheckpointAdjoint` (reverse-mode) or `ForwardMode` (forward-mode AD)
- **Max steps**: 16 384 per integration call

JAX tracing allows the full forward pass—including the ODE—to be differentiated
with respect to `theta`, enabling gradient-based optimisation.
